# News Multi-Label Classification — DL решение

In [ ]:
# !pip install transformers torch datasets scikit-learn pandas numpy matplotlib seaborn --quiet

In [ ]:
import os, re, ast, random, warnings
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

SEED = 322
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = (DEVICE.type == 'cuda')

print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU:   {torch.cuda.get_device_name(0)}')
    print(f'VRAM:  {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
print(f'AMP:   {USE_AMP}')
print('Imports OK')

## 1. Загрузка данных

In [ ]:
train = pd.read_csv('train.csv', sep='\t')
test  = pd.read_csv('test.csv',  sep='\t')

train['target_list'] = train['target'].apply(ast.literal_eval)
Y_all = np.array(train['target_list'].tolist(), dtype=np.float32)

LABEL_NAMES = ['Politics', 'Economics', 'Culture', 'Tech/Science', 'Sports']
N_LABELS    = len(LABEL_NAMES)
MODEL_NAME  = 'DeepPavlov/rubert-base-cased'

print(f'Train: {train.shape}  Test: {test.shape}')
print('\nМетки:')
for i, name in enumerate(LABEL_NAMES):
    pos = int(Y_all[:, i].sum())
    print(f'  {i} {name:<14}: {pos:5d} ({pos/len(Y_all)*100:.1f}%)')
print(f'\nНулевые строки (нет метки): {(Y_all.sum(1)==0).sum()} ({(Y_all.sum(1)==0).mean()*100:.1f}%)')
print('\nTrain sources:', train['source'].value_counts().to_dict())
print('Test  sources:', test['source'].value_counts().to_dict())

## 2. EDA

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

train['source'].value_counts().plot.bar(
    ax=axes[0], title='Train — источники', rot=0, color='steelblue')
test['source'].value_counts().plot.bar(
    ax=axes[1], title='Test — источники', rot=0, color='coral')

axes[2].bar(LABEL_NAMES, Y_all.sum(axis=0), color='seagreen')
axes[2].set_title('Примеров на метку')
axes[2].tick_params(axis='x', rotation=15)

plt.tight_layout(); plt.show()
print('Domain shift: Spletnesti и Zholtosti только в test!')

In [ ]:
cooc = Y_all.T @ Y_all
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cooc.astype(int), annot=True, fmt='d',
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, cmap='YlOrRd', ax=axes[0])
axes[0].set_title('Co-occurrence меток')

pd.Series(Y_all.sum(axis=1)).value_counts().sort_index().plot.bar(
    ax=axes[1], title='Меток на пример', color='steelblue', rot=0)
plt.tight_layout(); plt.show()

In [ ]:
# Демонстрация зашумлённости по каждому источнику
import re as _r
def noise_summary(df, src):
    texts = df[df['source'] == src]['text'].fillna('').tolist()
    joined = ' '.join(texts)
    emoji4   = len(_r.findall(r'[\U00010000-\U0010FFFF]', joined))
    sym3     = len(_r.findall(r'[☀-⟿]', joined))
    html     = len(_r.findall(r'<[^>]+>', joined))
    entities = len(_r.findall(r'&[a-zA-Z#][a-zA-Z0-9]{1,6};', joined))
    nbsp     = joined.count('\xa0')
    multi_ws = len(_r.findall(r' {5,}', joined))
    print(f'{src:<14}: emoji={emoji4:6d}  sym={sym3:5d}  html={html:6d}  ent={entities:5d}  nbsp={nbsp:6d}  multi_ws={multi_ws:5d}')

print('Мусор по источникам:')
for src in ['Novosti', 'Svezhesti']:
    noise_summary(train, src)
print('---')
for src in ['Novosti', 'Svezhesti', 'Zholtosti', 'Spletnesti']:
    noise_summary(test, src)

print()
print('Примеры текстов:')
print('Train/Novosti:   ', repr(train[train['source']=='Novosti']['text'].iloc[0][:200]))
print()
print('Test/Novosti:    ', repr(test[test['source']=='Novosti']['text'].iloc[0][:200]))
print()
print('Test/Zholtosti:  ', repr(test[test['source']=='Zholtosti']['text'].iloc[0][:200]))
print()
print('Test/Spletnesti: ', repr(test[test['source']=='Spletnesti']['text'].iloc[0][:200]))

In [ ]:
# Длина текстов в токенах (приблизительно: слова * 1.5 для русского)
train['raw_words'] = train['text'].fillna('').str.split().str.len()
test['raw_words']  = test['text'].fillna('').str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
train['raw_words'].clip(upper=600).plot.hist(bins=60, ax=axes[0],
    title='Train — длина текста (слов)', color='steelblue', edgecolor='none')
axes[0].axvline(330, color='red', linestyle='--', label='≈512 токенов')
axes[0].legend()
test['raw_words'].clip(upper=600).plot.hist(bins=60, ax=axes[1],
    title='Test — длина текста (слов)', color='coral', edgecolor='none')
axes[1].axvline(330, color='red', linestyle='--', label='≈512 токенов')
axes[1].legend()
plt.tight_layout(); plt.show()

over = (train['raw_words'] > 330).mean()
print(f'Train: {over*100:.0f}% текстов длиннее 512 токенов — будем обрезать')

## 3. Препроцессинг

In [ ]:
import html as _html

_BROKEN_ENTITY = re.compile(
    r'\b(lt|gt|amp|apos|quot|nbsp|mdash|ndash|laquo|raquo|hellip|'
    r'ldquo|rdquo|lsquo|rsquo|copy|reg|oacute|eacute|ccedil|ntilde|'
    r'#\d{1,6}|#x[0-9a-fA-F]{1,6})\s*;',
    re.IGNORECASE,
)


def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = re.sub(r'<!\[CDATA\[.*?\]\]>', ' ', text, flags=re.DOTALL)
    text = re.sub(r'<!--.*?-->', ' ', text, flags=re.DOTALL)
    text = re.sub(r'<\?xml[^>]*\?>', ' ', text)
    text = re.sub(r'<(script|style)[^>]*>.*?</\1>', ' ', text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r'\(function\s*\(\)\s*\{.*?\}\s*\)\s*;?', ' ', text, flags=re.DOTALL)
    text = re.sub(r'VK\.Widgets\.\w+\([^)]*\)', ' ', text)
    text = re.sub(r'<[^<>]{0,100}>', ' ', text)
    text = re.sub(r'<[^<>]{0,100}>', ' ', text)
    text = re.sub(r'[<>]', ' ', text)
    for _ in range(3):
        text = _html.unescape(text)
    text = re.sub(r'[<>]', ' ', text)
    text = _BROKEN_ENTITY.sub(' ', text)
    text = re.sub(r'&\S{0,12};?', ' ', text)
    text = re.sub(r'https?://\S+', ' ', text)
    text = text.replace('\xa0', ' ')
    text = re.sub(r'[\U00010000-\U0010FFFF]', ' ', text)
    text = re.sub(r'[⌀-⯿]', ' ', text)
    text = re.sub(r'[︀-﻿]', ' ', text)
    text = re.sub(r'[©®\xad⠀]', ' ', text)
    text = re.sub(r'[​-‍\u202C⁣]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


train['title_clean'] = train['title'].apply(clean_text)
train['text_clean']  = train['text'].apply(clean_text)
test['title_clean']  = test['title'].apply(clean_text)
test['text_clean']   = test['text'].apply(clean_text)

_emoji_pat  = re.compile(r'[\U00010000-\U0010FFFF⌀-⯿︀-﻿©®\xad]')
_entity_pat = re.compile(r'\b(lt|gt|amp|nbsp|mdash|ndash|laquo|raquo|hellip|ldquo|rdquo|lsquo|rsquo|copy|reg)\b')
_tag_pat    = re.compile(r'[<>&]')
_js_pat     = re.compile(r'function\s*\(|VK\.Widgets')

for name, df in [('TRAIN', train), ('TEST', test)]:
    e  = sum(1 for t in df['text_clean'] if _emoji_pat.search(t))
    tg = sum(1 for t in df['text_clean'] if _tag_pat.search(t))
    en = sum(1 for t in df['text_clean'] if _entity_pat.search(t))
    js = sum(1 for t in df['text_clean'] if _js_pat.search(t))
    print(f'{name}: мусор-символы={e}  <>&={tg}  entity-обрывки={en}  JS={js}')
print()

for split, df, src in [
    ('Train', train, 'Novosti'), ('Train', train, 'Svezhesti'),
    ('Test',  test,  'Novosti'), ('Test',  test,  'Svezhesti'),
    ('Test',  test,  'Zholtosti'), ('Test',  test,  'Spletnesti'),
]:
    row = df[df['source'] == src].iloc[0]
    print(f'--- {split}/{src} ---')
    print(f'Title: {row["title_clean"][:80]}')
    print(f'Text:  {row["text_clean"][:250]}')
    print()

train['clean_words'] = train['text_clean'].str.split().str.len()
test['clean_words']  = test['text_clean'].str.split().str.len()
print('Длина (слов) после очистки:')
print('Train:', train['clean_words'].describe().astype(int)[['mean','50%','75%','max']].to_dict())
print('Test: ', test['clean_words'].describe().astype(int)[['mean','50%','75%','max']].to_dict())

## 4. Dataset и токенизация

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LEN = 512

print(f'Tokenizer: {MODEL_NAME}')
print(f'Vocab size: {tokenizer.vocab_size}')

# Пример токенизации
t = train['title_clean'].iloc[0]
b = train['text_clean'].iloc[0]
enc = tokenizer(t + ' ' + t + ' ' + t, b,
                max_length=MAX_LEN, truncation='only_second',
                padding='max_length', return_tensors='pt')
print(f'Пример input_ids shape: {enc["input_ids"].shape}')
# Сколько токенов занял заголовок×3
sep_pos = (enc['input_ids'][0] == tokenizer.sep_token_id).nonzero(as_tuple=True)[0]
print(f'Позиции [SEP]: {sep_pos.tolist()}  (заголовок×3 занял ~{sep_pos[0].item()} токенов)')

In [ ]:
class NewsDataset(Dataset):
    def __init__(self, titles, texts, labels=None, tokenizer=None, max_len=512):
        self.titles    = titles
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.titles)

    def __getitem__(self, idx):
        title3 = ' '.join([str(self.titles[idx])] * 3)
        text   = str(self.texts[idx])

        enc = self.tokenizer(
            title3, text,
            max_length=self.max_len,
            truncation='only_second',
            padding='max_length',
            return_tensors='pt',
        )
        item = {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
        }
        if 'token_type_ids' in enc:
            item['token_type_ids'] = enc['token_type_ids'].squeeze(0)
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float32)
        return item

## 5. Архитектура модели

`rubert-base-cased` → mean pooling последнего слоя (устойчивее CLS для длинных текстов)  
→ Dropout(0.1) → Linear(768 → 256) → GELU → Dropout(0.1) → Linear(256 → 5)

In [ ]:
class MultiLabelBERT(nn.Module):
    def __init__(self, model_name, n_labels, dropout=0.1):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden    = self.bert.config.hidden_size
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, n_labels),
        )

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        out    = self.bert(input_ids=input_ids, attention_mask=attention_mask,
                           token_type_ids=token_type_ids)
        mask   = attention_mask.unsqueeze(-1).float()
        pooled = (out.last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        return self.head(pooled)


_m = MultiLabelBERT(MODEL_NAME, N_LABELS)
print(f'Параметров: {sum(p.numel() for p in _m.parameters()):,}')
print(f'hidden={_m.bert.config.hidden_size}  layers={_m.bert.config.num_hidden_layers}')
del _m

## 6. Train/Val split и DataLoader-ы

In [ ]:
stratum = np.full(len(train), N_LABELS, dtype=int)
for i in range(len(train)):
    active = np.where(Y_all[i] == 1)[0]
    if len(active):
        stratum[i] = active[0]

tr_idx, val_idx = train_test_split(
    np.arange(len(train)),
    test_size=0.1,
    stratify=stratum,
    random_state=SEED,
)
print(f'Train: {len(tr_idx)}  Val: {len(val_idx)}')

BATCH_SIZE  = 16
GRAD_ACCUM  = 2
EPOCHS      = 5
LR          = 2e-5
WARMUP_FRAC = 0.1

print(f'batch={BATCH_SIZE}  grad_accum={GRAD_ACCUM}  effective_batch={BATCH_SIZE*GRAD_ACCUM}')
print(f'epochs={EPOCHS}  lr={LR}  warmup={WARMUP_FRAC}  amp={USE_AMP}')

In [ ]:
train_titles = train['title_clean'].values
train_texts  = train['text_clean'].values

train_dataset = NewsDataset(
    train_titles[tr_idx], train_texts[tr_idx],
    labels=Y_all[tr_idx], tokenizer=tokenizer, max_len=MAX_LEN,
)
val_dataset = NewsDataset(
    train_titles[val_idx], train_texts[val_idx],
    labels=Y_all[val_idx], tokenizer=tokenizer, max_len=MAX_LEN,
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE * 2, shuffle=False,
                          num_workers=4, pin_memory=True)

print(f'Train batches: {len(train_loader)}  Val batches: {len(val_loader)}')

## 7. Утилиты обучения

In [ ]:
def compute_pos_weight(Y, indices):
    pos = Y[indices].sum(axis=0)
    neg = len(indices) - pos
    return torch.tensor(neg / np.maximum(pos, 1), dtype=torch.float32)


def train_epoch(model, loader, optimizer, scheduler, criterion, scaler, device,
                grad_accum=1, epoch=None, n_epochs=None):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()

    desc = f'Train {epoch}/{n_epochs}' if epoch else 'Train'
    pbar = tqdm(loader, desc=desc, leave=True, dynamic_ncols=True)

    for step, batch in enumerate(pbar):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        token_type_ids = batch.get('token_type_ids')
        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(device)
        labels = batch['labels'].to(device)

        with autocast(enabled=USE_AMP):
            logits = model(input_ids, attention_mask, token_type_ids)
            loss   = criterion(logits, labels) / grad_accum

        scaler.scale(loss).backward()

        if (step + 1) % grad_accum == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * grad_accum
        pbar.set_postfix({'loss': f'{total_loss / (step + 1):.4f}',
                          'lr':   f'{scheduler.get_last_lr()[0]:.2e}'})

    return total_loss / len(loader)


@torch.no_grad()
def predict_proba(model, loader, device, desc='Eval'):
    model.eval()
    all_probs = []
    pbar = tqdm(loader, desc=desc, leave=False, dynamic_ncols=True)
    for batch in pbar:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        token_type_ids = batch.get('token_type_ids')
        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(device)
        with autocast(enabled=USE_AMP):
            logits = model(input_ids, attention_mask, token_type_ids)
        probs = torch.sigmoid(logits).float().cpu().numpy()
        all_probs.append(probs)
    return np.vstack(all_probs)


print('Утилиты определены')

## 8. Обучение (val split)

In [ ]:
THRESHOLD = 0.5

model      = MultiLabelBERT(MODEL_NAME, N_LABELS, dropout=0.1).to(DEVICE)
pos_weight = compute_pos_weight(Y_all, tr_idx).to(DEVICE)
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
scaler     = GradScaler(enabled=USE_AMP)

optimizer = torch.optim.AdamW([
    {'params': list(model.bert.parameters()), 'lr': LR},
    {'params': list(model.head.parameters()), 'lr': LR * 10},
], weight_decay=0.01)

total_steps  = (len(train_loader) // GRAD_ACCUM) * EPOCHS
warmup_steps = int(total_steps * WARMUP_FRAC)
scheduler    = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

print(f'pos_weight: {pos_weight.cpu().numpy().round(2)}')
print(f'steps: {total_steps}  warmup: {warmup_steps}')
print()

history     = {'train_loss': [], 'val_micro_f1': [], 'val_macro_f1': []}
best_val_f1 = 0.0

epoch_bar = tqdm(range(1, EPOCHS + 1), desc='Epochs', unit='ep')
for epoch in epoch_bar:
    tr_loss   = train_epoch(model, train_loader, optimizer, scheduler,
                            criterion, scaler, DEVICE, GRAD_ACCUM,
                            epoch=epoch, n_epochs=EPOCHS)
    val_probs = predict_proba(model, val_loader, DEVICE, desc='Val')
    val_pred  = (val_probs >= THRESHOLD).astype(int)

    micro = f1_score(Y_all[val_idx], val_pred, average='micro', zero_division=0)
    macro = f1_score(Y_all[val_idx], val_pred, average='macro', zero_division=0)

    history['train_loss'].append(tr_loss)
    history['val_micro_f1'].append(micro)
    history['val_macro_f1'].append(macro)

    flag = ''
    if micro > best_val_f1:
        best_val_f1 = micro
        torch.save(model.state_dict(), 'best_model.pt')
        flag = ' *'

    epoch_bar.set_postfix({'loss': f'{tr_loss:.4f}', 'micro_f1': f'{micro:.4f}', 'best': f'{best_val_f1:.4f}'})
    tqdm.write(f'Epoch {epoch}/{EPOCHS}  loss={tr_loss:.4f}  micro_f1={micro:.4f}  macro_f1={macro:.4f}{flag}')

print(f'\nЛучший Val Micro F1: {best_val_f1:.4f}')

## 9. Анализ обучения и ошибок

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ep = range(1, EPOCHS + 1)

axes[0].plot(ep, history['train_loss'], marker='o', color='steelblue')
axes[0].set_title('Train Loss'); axes[0].set_xlabel('Epoch')

axes[1].plot(ep, history['val_micro_f1'], marker='o', label='Micro F1')
axes[1].plot(ep, history['val_macro_f1'], marker='s', label='Macro F1')
axes[1].set_title('Val F1'); axes[1].set_xlabel('Epoch'); axes[1].legend()

plt.tight_layout(); plt.show()

In [ ]:
model.load_state_dict(torch.load('best_model.pt', map_location=DEVICE))
val_probs = predict_proba(model, val_loader, DEVICE)
val_pred  = (val_probs >= THRESHOLD).astype(int)

print(f'Best Val Micro F1: {f1_score(Y_all[val_idx], val_pred, average="micro", zero_division=0):.4f}')
print(f'Best Val Macro F1: {f1_score(Y_all[val_idx], val_pred, average="macro", zero_division=0):.4f}')
print()
print('Per-label F1:')
for i, name in enumerate(LABEL_NAMES):
    f1 = f1_score(Y_all[val_idx, i], val_pred[:, i], zero_division=0)
    print(f'  {name:<14}: F1={f1:.4f}')

In [ ]:
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, N_LABELS, figsize=(18, 3))
for i, name in enumerate(LABEL_NAMES):
    cm = confusion_matrix(Y_all[val_idx, i], val_pred[:, i])
    sns.heatmap(cm, annot=True, fmt='d', ax=axes[i], cmap='Blues',
                xticklabels=['Pred 0', 'Pred 1'],
                yticklabels=['True 0', 'True 1'])
    axes[i].set_title(name)
plt.suptitle('Confusion matrix (val)', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, N_LABELS, figsize=(18, 3))
for i, name in enumerate(LABEL_NAMES):
    pos_mask = Y_all[val_idx, i] == 1
    axes[i].hist(val_probs[pos_mask,  i], bins=30, alpha=0.6, color='green', label='pos')
    axes[i].hist(val_probs[~pos_mask, i], bins=30, alpha=0.6, color='red',   label='neg')
    axes[i].axvline(THRESHOLD, color='black', linestyle='--', label=f'thr={THRESHOLD}')
    axes[i].set_title(name); axes[i].legend(fontsize=7)
plt.suptitle('Вероятности pos/neg (val)', y=1.02)
plt.tight_layout(); plt.show()

## 10. Финальная модель — всё train

In [ ]:
full_dataset = NewsDataset(
    train['title_clean'].values, train['text_clean'].values,
    labels=Y_all, tokenizer=tokenizer, max_len=MAX_LEN,
)
full_loader = DataLoader(full_dataset, batch_size=BATCH_SIZE, shuffle=True,
                         num_workers=4, pin_memory=True)
print(f'Full train batches: {len(full_loader)}')

In [ ]:
model_full      = MultiLabelBERT(MODEL_NAME, N_LABELS, dropout=0.1).to(DEVICE)
pos_weight_full = compute_pos_weight(Y_all, np.arange(len(train))).to(DEVICE)
criterion_full  = nn.BCEWithLogitsLoss(pos_weight=pos_weight_full)
scaler_full     = GradScaler(enabled=USE_AMP)

optimizer_full = torch.optim.AdamW([
    {'params': list(model_full.bert.parameters()), 'lr': LR},
    {'params': list(model_full.head.parameters()), 'lr': LR * 10},
], weight_decay=0.01)

total_full  = (len(full_loader) // GRAD_ACCUM) * EPOCHS
warmup_full = int(total_full * WARMUP_FRAC)
sched_full  = get_cosine_schedule_with_warmup(optimizer_full, warmup_full, total_full)

print(f'Финальное обучение: {EPOCHS} эпох, {total_full} шагов')

epoch_bar = tqdm(range(1, EPOCHS + 1), desc='Epochs (full)', unit='ep')
for epoch in epoch_bar:
    loss = train_epoch(model_full, full_loader, optimizer_full, sched_full,
                       criterion_full, scaler_full, DEVICE, GRAD_ACCUM,
                       epoch=epoch, n_epochs=EPOCHS)
    epoch_bar.set_postfix({'loss': f'{loss:.4f}'})
    tqdm.write(f'Epoch {epoch}/{EPOCHS}  loss={loss:.4f}')

print('\nФинальная модель обучена')

## 11. Инференс и постпроцессинг

In [ ]:
test_dataset = NewsDataset(
    test['title_clean'].values, test['text_clean'].values,
    labels=None, tokenizer=tokenizer, max_len=MAX_LEN,
)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE * 2, shuffle=False,
                         num_workers=4, pin_memory=True)

test_probs = predict_proba(model_full, test_loader, DEVICE)
Y_pred     = (test_probs >= THRESHOLD).astype(int)

print(f'Test probs shape: {test_probs.shape}')
print(f'\nРаспределение предсказанных меток (test, thr={THRESHOLD}):')
for i, name in enumerate(LABEL_NAMES):
    cnt = Y_pred[:, i].sum()
    print(f'  {name:<14}: {cnt:5d} ({cnt/len(test)*100:.1f}%)')
print(f'Нет меток: {(Y_pred.sum(1)==0).sum()} ({(Y_pred.sum(1)==0).mean()*100:.1f}%)')

In [ ]:
fig, axes = plt.subplots(1, N_LABELS, figsize=(18, 3))
for i, name in enumerate(LABEL_NAMES):
    axes[i].hist(test_probs[:, i], bins=50, color='steelblue', edgecolor='none')
    axes[i].axvline(THRESHOLD, color='red', linestyle='--', label=f'thr={THRESHOLD}')
    axes[i].set_title(name); axes[i].legend(fontsize=8)
plt.suptitle('Распределение вероятностей (test)', y=1.02)
plt.tight_layout(); plt.show()

print('\nПо источникам (test):')
print(f'{"Source":<14}  ' + '  '.join(f'{n[:7]:>8}' for n in LABEL_NAMES))
print('-' * 65)
for src in sorted(test['source'].unique()):
    mask  = (test['source'] == src).values
    means = Y_pred[mask].mean(axis=0)
    print(f'{src:<14}  ' + '  '.join(f'{m:>8.3f}' for m in means))

## 12. Сохранение сабмита

In [ ]:
submission = pd.DataFrame({
    'id':     test['id'].values,
    'target': ['[' + ','.join(str(v) for v in Y_pred[i]) + ']'
               for i in range(len(test))],
})
submission.to_csv('sample_submission.csv', index=False)
print(f'sample_submission.csv сохранён: {submission.shape}')
submission.head(10)

In [ ]:
import re as _re
assert len(submission) == len(test)
assert list(submission.columns) == ['id', 'target']
assert (submission['id'].values == test['id'].values).all()
assert submission['target'].apply(
    lambda x: bool(_re.match(r'^\[([01],){4}[01]\]$', x))
).all()
print('Все проверки пройдены!')